# Set up

## Imports

In [ ]:
import numpy as np
import pandas as pd
import json

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## Load and split data

In [ ]:
df = pd.read_parquet("../data/brfss2025_il_clean.parquet")
with open("../data/brfss2025_il_clean_columns.json") as f:
    meta = json.load(f)
cat_predictors, num_predictors = meta["cat_predictors"], meta["num_predictors"]

assert df["sleep_cat"].cat.categories.tolist() == ["normal", "short", "long"]

X = df[cat_predictors + num_predictors]
y = df[meta["target"]]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


## Preprocess

In [ ]:
preprocessor = ColumnTransformer([
    ("cat", make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(handle_unknown="ignore", drop="if_binary")),
    cat_predictors),
    ("num", make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True),
        StandardScaler()),
    num_predictors),
])

- Consider log transform on drinks_per_week and diet columns when doing logistic regression, very right skewed (np.log1p)
- 19% of rows have a missing value, experiment with models that drop these rows